# Notebook 07 — Real-World Holdout Validation

Final validation of the LightGBM model against genuinely unseen, real-world data — not a CV fold from the original 2019–2023 dataset, but live station readings pulled from the OpenAQ API, spanning February 2025 through June/July 2026 depending on city.

Models were retrained on the full original dataset (through March 2023) and evaluated against this new window with no overlap with training data.

## Data Source & Station Matching

Original CPCB CCR portal access was attempted but proved non-functional (broken advanced-search UI). OpenAQ was used instead, matched by proximity to each original Kaggle station:

| City | Original Station | OpenAQ Match | Exact Match? |
|------|-------------------|--------------|---------------|
| Mumbai | Sion (MH015) | Sion, Mumbai - MPCB | ✅ |
| Chennai | Velachery Res. Area (TN003) | Velachery Res. Area, Chennai - CPCB | ✅ |
| Hyderabad | Bollaram Industrial Area (TG004) | Bollaram Industrial Area, Hyderabad - TSPCB | ✅ |
| Bengaluru | Hebbal (KA008) | Hebbal, Bengaluru - KSPCB | ✅ |
| Delhi | Dwarka Sector 8 (DL030) | NSIT Dwarka, Delhi - CPCB | ⚠️ Nearest available, not exact match |

Each station had two PM2.5 sensor generations; the newer sensor (reporting Feb 2025 onward) was used for all cities. A ~2.3-year reporting gap exists between the old and new sensors at every station (Oct 2022 – Feb 2025), so the holdout window is treated as a self-contained period with features engineered fresh (not concatenated to the original training series).

## Data Quality: Mumbai Sensor Malfunction

Mumbai's OpenAQ feed showed a cluster of implausible PM2.5 readings (609–985 µg/m³) between Sept 25 – Oct 9, 2025 — during monsoon season, when Mumbai's pollution is typically at its calmest per EDA findings. The values decayed in a near-monotonic staircase pattern over two weeks, consistent with a stuck or drifting sensor rather than a genuine air quality event (no comparably extreme reading was ever observed in the original 4-year training set, even in Delhi's worst winter smog days). This window was excluded from evaluation.

## Holdout Results

| City | Original CV MAE | Holdout MAE | Holdout Window |
|------|-----------------|-------------|-----------------|
| Delhi | 25.74 | 22.41 | Feb 2025 – Jun 2026 |
| Mumbai | 10.13 | 23.44 | Feb 2025 – Jun 2026 (malfunction window removed) |
| Chennai | 6.60 | 5.25 | Feb 2025 – Jun 2026 |
| Hyderabad | 7.29 | 6.95 | Feb 2025 – Mar 2026 |
| Bengaluru | 7.45 | 7.60 | Feb 2025 – Jun 2026 |

**Chennai, Hyderabad, and Bengaluru matched or beat their original cross-validated performance on genuinely unseen future data** — strong evidence the model generalizes beyond the folds it was validated on. Delhi held up well despite the station mismatch.

**Mumbai remains an open limitation**: even after removing the confirmed sensor malfunction window, holdout MAE is roughly 2.3x higher than CV MAE, driven by a small number of isolated large misses (a Dec 27 2025 spike plausibly tied to New Year celebrations, plus two unexplained spring 2026 spikes) that the model could not anticipate. This is reported as a genuine finding rather than resolved further.

In [1]:
from google.colab import userdata
import requests

API_KEY = userdata.get('OPENAQ_API_KEY')
headers = {'X-API-Key': API_KEY}

city_coords = {
    'Delhi': (28.6, 77.05),
    'Mumbai': (19.04, 72.86),
    'Chennai': (12.98, 80.22),
    'Hyderabad': (17.5, 78.4),
    'Bengaluru': (13.05, 77.59),
}

for city, (lat, lon) in city_coords.items():
    resp = requests.get(
        'https://api.openaq.org/v3/locations',
        headers=headers,
        params={'coordinates': f'{lat},{lon}', 'radius': 15000, 'limit': 5}
    )
    print(city, resp.status_code)
    print(resp.json())
    print()

Delhi 200
{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 5, 'found': '>5'}, 'results': [{'id': 15, 'name': 'IGI Airport', 'locality': None, 'timezone': 'Asia/Kolkata', 'country': {'id': 9, 'code': 'IN', 'name': 'India'}, 'owner': {'id': 4, 'name': 'Unknown Governmental Organization'}, 'provider': {'id': 168, 'name': 'CPCB'}, 'isMobile': False, 'isMonitor': True, 'instruments': [{'id': 2, 'name': 'Government Monitor'}], 'sensors': [{'id': 27, 'name': 'co µg/m³', 'parameter': {'id': 4, 'name': 'co', 'units': 'µg/m³', 'displayName': 'CO mass'}}, {'id': 28, 'name': 'no2 µg/m³', 'parameter': {'id': 5, 'name': 'no2', 'units': 'µg/m³', 'displayName': 'NO₂ mass'}}, {'id': 29, 'name': 'o3 µg/m³', 'parameter': {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': 'O₃ mass'}}, {'id': 26, 'name': 'pm10 µg/m³', 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}, {'id': 30, 'name': 'pm25 µg/m³', 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg

In [2]:
missing_cities = {
    'Delhi': (28.6, 77.05),      # looking for Dwarka Sector 8
    'Mumbai': (19.04, 72.86),    # looking for Sion
    'Bengaluru': (13.05, 77.59), # looking for Hebbal
}

for city, (lat, lon) in missing_cities.items():
    resp = requests.get(
        'https://api.openaq.org/v3/locations',
        headers=headers,
        params={'coordinates': f'{lat},{lon}', 'radius': 25000, 'limit': 15}
    )
    results = resp.json()['results']
    print(f"--- {city} ---")
    for r in results:
        print(r['id'], r['name'])
    print()

--- Delhi ---
13 Delhi Technological University, Delhi - CPCB
15 IGI Airport
16 Civil Lines
17 R K Puram, Delhi - DPCC
50 Punjabi Bagh, Delhi - DPCC
103 Income Tax Office, Delhi - CPCB
236 Mandir Marg, Delhi - DPCC
301 Vikas Sadan, Gurugram - HSPCB
2503 Shadipur, Delhi - CPCB
2597 US Diplomatic Post: New Delhi
5404 Pusa, Delhi - IMD
5540 Punjabi Bagh, Delhi - DPCC
5541 Burari Crossing, New Delhi - IMD
5570 Aya Nagar, New Delhi - IMD
5581 Pusa, New Delhi - IMD

--- Mumbai ---
2598 Navi Mumbai Municipal Corporation Airoli
5593 Pimpleshwar Mandir, Thane - MPCB
6927 Colaba, Mumbai - MPCB
6943 Mahape, Navi Mumbai - MPCB
6945 Kurla, Mumbai - MPCB
6948 Chhatrapati Shivaji Intl. Airport (T2), Mumbai - MPCB
6956 Powai, Mumbai - MPCB
6959 Siddharth Nagar-Worli, Mumbai - IITM
6965 Borivali East, Mumbai - MPCB
6967 Sion, Mumbai - MPCB
6987 Vile Parle West, Mumbai - MPCB
7850 Bandra, Mumbai - MPCB
8039 Mumbai
11602 Bandra Kurla Complex, Mumbai - IITM
11604 Deonar, Mumbai - IITM

--- Bengaluru ---
4

In [5]:
resp = requests.get(
    'https://api.openaq.org/v3/locations',
    headers=headers,
    params={'coordinates': '28.59,77.05', 'radius': 25000, 'limit': 25}
)
results = resp.json()['results']
for r in results:
    print(r['id'], r['name'], r['coordinates'])

13 Delhi Technological University, Delhi - CPCB {'latitude': 28.744, 'longitude': 77.12}
15 IGI Airport {'latitude': 28.56, 'longitude': 77.094}
16 Civil Lines {'latitude': 28.6787, 'longitude': 77.2262}
17 R K Puram, Delhi - DPCC {'latitude': 28.563262, 'longitude': 77.186937}
50 Punjabi Bagh, Delhi - DPCC {'latitude': 28.674045, 'longitude': 77.131023}
103 Income Tax Office, Delhi - CPCB {'latitude': 28.623499999999996, 'longitude': 77.2494}
236 Mandir Marg, Delhi - DPCC {'latitude': 28.6341, 'longitude': 77.2005}
301 Vikas Sadan, Gurugram - HSPCB {'latitude': 28.4501238, 'longitude': 77.0263051}
2503 Shadipur, Delhi - CPCB {'latitude': 28.651699999999998, 'longitude': 77.1581}
2597 US Diplomatic Post: New Delhi {'latitude': 28.598096, 'longitude': 77.189066}
5404 Pusa, Delhi - IMD {'latitude': 28.639645, 'longitude': 77.146263}
5540 Punjabi Bagh, Delhi - DPCC {'latitude': 28.6670856, 'longitude': 77.1301247}
5541 Burari Crossing, New Delhi - IMD {'latitude': 28.7256504, 'longitude':

In [6]:
location_ids = {
    'Delhi': 5622,
    'Mumbai': 6967,
    'Chennai': 5655,
    'Hyderabad': 5599,
    'Bengaluru': 6984,
}

for city, loc_id in location_ids.items():
    resp = requests.get(
        f'https://api.openaq.org/v3/locations/{loc_id}',
        headers=headers
    )
    location = resp.json()['results'][0]
    pm25_sensors = [s for s in location['sensors'] if s['parameter']['name'] == 'pm25']
    print(f"--- {city} (location {loc_id}) ---")
    for s in pm25_sensors:
        print(f"  sensor id: {s['id']}")
    print()

--- Delhi (location 5622) ---
  sensor id: 14922
  sensor id: 12234744

--- Mumbai (location 6967) ---
  sensor id: 20096
  sensor id: 12243930

--- Chennai (location 5655) ---
  sensor id: 12235531
  sensor id: 15136

--- Hyderabad (location 5599) ---
  sensor id: 12235400
  sensor id: 15217

--- Bengaluru (location 6984) ---
  sensor id: 20212
  sensor id: 12235249



In [7]:
sensor_candidates = {
    'Delhi': [14922, 12234744],
    'Mumbai': [20096, 12243930],
    'Chennai': [12235531, 15136],
    'Hyderabad': [12235400, 15217],
    'Bengaluru': [20212, 12235249],
}

for city, sensor_ids in sensor_candidates.items():
    print(f"--- {city} ---")
    for sid in sensor_ids:
        resp = requests.get(f'https://api.openaq.org/v3/sensors/{sid}', headers=headers)
        s = resp.json()['results'][0]
        first = s.get('datetimeFirst', {}).get('local')
        last = s.get('datetimeLast', {}).get('local')
        print(f"  sensor {sid}: {first} to {last}")
    print()

--- Delhi ---
  sensor 14922: 2018-03-09T11:30:00+05:30 to 2022-10-31T07:30:00+05:30
  sensor 12234744: 2025-02-19T01:45:00+05:30 to 2026-07-01T18:00:00+05:30

--- Mumbai ---
  sensor 20096: 2019-06-27T10:45:00+05:30 to 2022-10-31T07:15:00+05:30
  sensor 12243930: 2025-02-19T15:00:00+05:30 to 2026-07-01T16:30:00+05:30

--- Chennai ---
  sensor 12235531: 2025-02-19T01:45:00+05:30 to 2026-07-01T18:00:00+05:30
  sensor 15136: 2018-03-09T11:00:00+05:30 to 2022-10-31T07:15:00+05:30

--- Hyderabad ---
  sensor 12235400: 2025-02-19T01:45:00+05:30 to 2026-03-02T12:30:00+05:30
  sensor 15217: 2018-03-09T11:30:00+05:30 to 2022-10-31T07:30:00+05:30

--- Bengaluru ---
  sensor 20212: 2018-08-10T11:30:00+05:30 to 2022-10-31T07:30:00+05:30
  sensor 12235249: 2025-02-19T01:45:00+05:30 to 2026-07-01T17:00:00+05:30



In [9]:
import pandas as pd

In [10]:
newer_sensors = {
    'Delhi': 12234744,
    'Mumbai': 12243930,
    'Chennai': 12235531,
    'Hyderabad': 12235400,
    'Bengaluru': 12235249,
}

holdout_dfs = {}

for city, sensor_id in newer_sensors.items():
    resp = requests.get(
        f'https://api.openaq.org/v3/sensors/{sensor_id}/days',
        headers=headers,
        params={'limit': 1000}
    )
    data = resp.json()['results']

    rows = []
    for d in data:
        rows.append({
            'date': d['period']['datetimeFrom']['local'][:10],
            'pm25': d['value']
        })

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    holdout_dfs[city] = df
    print(f"{city}: {len(df)} days, {df['date'].min()} to {df['date'].max()}")

Delhi: 485 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00
Mumbai: 417 days, 2025-02-19 00:00:00 to 2026-06-29 00:00:00
Chennai: 456 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00
Hyderabad: 356 days, 2025-02-19 00:00:00 to 2026-03-02 00:00:00
Bengaluru: 466 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00


In [11]:
import numpy as np

# --- Log transform ---
for city, df in holdout_dfs.items():
    df['pm25_log'] = np.log1p(df['pm25'])

# --- Lags ---
lags = [1, 2, 3, 7, 14, 30, 365]
for city, df in holdout_dfs.items():
    for lag in lags:
        df[f'lag_{lag}'] = df['pm25_log'].shift(lag)

# --- Rolling stats ---
windows = [7, 14, 30]
for city, df in holdout_dfs.items():
    shifted = df['pm25_log'].shift(1)
    for w in windows:
        df[f'roll_mean_{w}'] = shifted.rolling(w).mean()
        df[f'roll_std_{w}'] = shifted.rolling(w).std()
    df['roll_min_7'] = shifted.rolling(7).min()
    df['roll_max_7'] = shifted.rolling(7).max()
    df['roll_min_30'] = shifted.rolling(30).min()
    df['roll_max_30'] = shifted.rolling(30).max()

# --- Calendar features ---
for city, df in holdout_dfs.items():
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# --- Holiday flags (2025-2026 dates, ±1 day buffer) ---
diwali_dates = pd.to_datetime([
    '2025-10-19', '2025-10-20', '2025-10-21',
])
republic_day_dates = pd.to_datetime([
    '2025-01-25', '2025-01-26', '2025-01-27',
    '2026-01-25', '2026-01-26', '2026-01-27',
])
holi_dates = pd.to_datetime([
    '2025-03-13', '2025-03-14', '2025-03-15',
    '2026-03-02', '2026-03-03', '2026-03-04',
])
new_year_dates = pd.to_datetime([
    '2024-12-31', '2025-01-01', '2025-01-02',
    '2025-12-31', '2026-01-01', '2026-01-02',
])

for city, df in holdout_dfs.items():
    df['is_diwali'] = df['date'].isin(diwali_dates).astype(int)
    df['is_republic_day'] = df['date'].isin(republic_day_dates).astype(int)
    df['is_holi'] = df['date'].isin(holi_dates).astype(int)
    df['is_new_year'] = df['date'].isin(new_year_dates).astype(int)

# --- Outlier flags ---
for city, df in holdout_dfs.items():
    mean_log = df['pm25_log'].mean()
    std_log = df['pm25_log'].std()
    z_score = (df['pm25_log'] - mean_log) / std_log
    df['is_outlier'] = (z_score.abs() > 3).astype(int)

holdout_dfs['Delhi'].head()

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,...,is_weekend,doy_sin,doy_cos,dow_sin,dow_cos,is_diwali,is_republic_day,is_holi,is_new_year,is_outlier
0,2025-02-19,82.0,4.418841,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0.757922,0.652346,0.974928,-0.222521,0,0,0,0,0
1,2025-02-20,64.3,4.178992,4.418841,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0.769031,0.639212,0.433884,-0.900969,0,0,0,0,0
2,2025-02-21,83.2,4.433195,4.178992,4.418841,NaN,NaN,NaN,NaN,NaN,...,0,0.779913,0.625889,-0.433884,-0.900969,0,0,0,0,0
3,2025-02-22,76.1,4.345103,4.433195,4.178992,4.418841,NaN,NaN,NaN,NaN,...,1,0.790563,0.612380,-0.974928,-0.222521,0,0,0,0,0
4,2025-02-23,76.1,4.345103,4.345103,4.433195,4.178992,NaN,NaN,NaN,NaN,...,1,0.800980,0.598691,-0.781831,0.623490,0,0,0,0,0


In [12]:
!git clone https://github.com/shaheeeeeeeeem/air-quality-forecasting.git

Cloning into 'air-quality-forecasting'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 43 (delta 12), reused 40 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 2.50 MiB | 11.65 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [30]:
import os
import requests
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from google.colab import userdata

In [31]:
repo_path = '/content/air-quality-forecasting/data/processed'

cities = ['Delhi', 'Mumbai', 'Chennai', 'Hyderabad', 'Bengaluru']

kaggle_dfs = {}

for city in cities:
    filepath = os.path.join(repo_path, f'{city}_features.csv')
    df = pd.read_csv(filepath, parse_dates=['date'])
    kaggle_dfs[city] = df

for city, df in kaggle_dfs.items():
    print(f"{city}: {df.shape[0]} rows, {df.shape[1]} columns")

Delhi: 1551 rows, 33 columns
Mumbai: 1381 rows, 33 columns
Chennai: 1551 rows, 33 columns
Hyderabad: 1551 rows, 33 columns
Bengaluru: 1551 rows, 33 columns


In [32]:
target_col = 'pm25_log'
exclude_cols = ['date', 'pm25', 'pm25_log']
feature_cols = [c for c in kaggle_dfs['Delhi'].columns if c not in exclude_cols]

In [33]:
newer_sensors = {
    'Delhi': 12234744,
    'Mumbai': 12243930,
    'Chennai': 12235531,
    'Hyderabad': 12235400,
    'Bengaluru': 12235249,
}

holdout_dfs = {}

for city, sensor_id in newer_sensors.items():
    resp = requests.get(
        f'https://api.openaq.org/v3/sensors/{sensor_id}/days',
        headers=headers,
        params={'limit': 1000}
    )
    data = resp.json()['results']
    rows = [{'date': d['period']['datetimeFrom']['local'][:10], 'pm25': d['value']} for d in data]
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    holdout_dfs[city] = df
    print(f"{city}: {len(df)} days, {df['date'].min()} to {df['date'].max()}")

Delhi: 485 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00
Mumbai: 417 days, 2025-02-19 00:00:00 to 2026-06-29 00:00:00
Chennai: 456 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00
Hyderabad: 356 days, 2025-02-19 00:00:00 to 2026-03-02 00:00:00
Bengaluru: 466 days, 2025-02-19 00:00:00 to 2026-06-30 00:00:00


In [34]:
for city, df in holdout_dfs.items():
    n_before = len(df)
    df.drop(df[df['pm25'] > 700].index, inplace=True)
    n_after = len(df)
    if n_before != n_after:
        print(f"{city}: dropped {n_before - n_after} implausible readings (>700 µg/m³)")

Delhi: dropped 1 implausible readings (>700 µg/m³)
Mumbai: dropped 6 implausible readings (>700 µg/m³)


In [35]:
for city, df in holdout_dfs.items():
    df['pm25'] = df['pm25'].interpolate(method='linear')

for city, df in holdout_dfs.items():
    print(f"{city}: {df['pm25'].isna().sum()} remaining NaNs")

Delhi: 0 remaining NaNs
Mumbai: 0 remaining NaNs
Chennai: 0 remaining NaNs
Hyderabad: 0 remaining NaNs
Bengaluru: 0 remaining NaNs


In [40]:
malfunction_start = pd.Timestamp('2025-09-25')
malfunction_end = pd.Timestamp('2025-10-09')

for city, df in holdout_dfs.items():
    if city == 'Mumbai':
        n_before = len(df)
        df.drop(df[(df['date'] >= malfunction_start) & (df['date'] <= malfunction_end)].index, inplace=True)
        print(f"{city}: dropped {n_before - len(df)} days from suspected sensor malfunction ({malfunction_start.date()} to {malfunction_end.date()})")

Mumbai: dropped 9 days from suspected sensor malfunction (2025-09-25 to 2025-10-09)


In [41]:
for city, df in holdout_dfs.items():
    df['pm25_log'] = np.log1p(df['pm25'])

lags = [1, 2, 3, 7, 14, 30, 365]
for city, df in holdout_dfs.items():
    for lag in lags:
        df[f'lag_{lag}'] = df['pm25_log'].shift(lag)

windows = [7, 14, 30]
for city, df in holdout_dfs.items():
    shifted = df['pm25_log'].shift(1)
    for w in windows:
        df[f'roll_mean_{w}'] = shifted.rolling(w).mean()
        df[f'roll_std_{w}'] = shifted.rolling(w).std()
    df['roll_min_7'] = shifted.rolling(7).min()
    df['roll_max_7'] = shifted.rolling(7).max()
    df['roll_min_30'] = shifted.rolling(30).min()
    df['roll_max_30'] = shifted.rolling(30).max()

for city, df in holdout_dfs.items():
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

diwali_dates = pd.to_datetime(['2025-10-19', '2025-10-20', '2025-10-21'])
republic_day_dates = pd.to_datetime(['2025-01-25', '2025-01-26', '2025-01-27',
                                       '2026-01-25', '2026-01-26', '2026-01-27'])
holi_dates = pd.to_datetime(['2025-03-13', '2025-03-14', '2025-03-15',
                               '2026-03-02', '2026-03-03', '2026-03-04'])
new_year_dates = pd.to_datetime(['2024-12-31', '2025-01-01', '2025-01-02',
                                   '2025-12-31', '2026-01-01', '2026-01-02'])

for city, df in holdout_dfs.items():
    df['is_diwali'] = df['date'].isin(diwali_dates).astype(int)
    df['is_republic_day'] = df['date'].isin(republic_day_dates).astype(int)
    df['is_holi'] = df['date'].isin(holi_dates).astype(int)
    df['is_new_year'] = df['date'].isin(new_year_dates).astype(int)

for city, df in holdout_dfs.items():
    mean_log = df['pm25_log'].mean()
    std_log = df['pm25_log'].std()
    z_score = (df['pm25_log'] - mean_log) / std_log
    df['is_outlier'] = (z_score.abs() > 3).astype(int)

In [42]:
holdout_results = []
trained_models = {}

for city, df in kaggle_dfs.items():
    X_train = df[feature_cols]
    y_train = df[target_col]

    model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=31,
                               random_state=42, verbosity=-1)
    model.fit(X_train, y_train)
    trained_models[city] = model

    holdout_df = holdout_dfs[city].copy()
    X_holdout = holdout_df[feature_cols]
    y_holdout_actual = holdout_df['pm25']

    preds_log = model.predict(X_holdout)
    preds_actual = np.expm1(preds_log)

    mae = mean_absolute_error(y_holdout_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_holdout_actual, preds_actual))

    holdout_results.append({
        'city': city, 'n_test_days': len(holdout_df),
        'test_start': holdout_df['date'].min(), 'test_end': holdout_df['date'].max(),
        'mae': mae, 'rmse': rmse
    })

holdout_results_df = pd.DataFrame(holdout_results)
holdout_results_df

,city,n_test_days,test_start,test_end,mae,rmse
0,Delhi,484,2025-02-19,2026-06-30,22.410770,37.858350
1,Mumbai,402,2025-02-19,2026-06-29,23.442471,51.662036
2,Chennai,456,2025-02-19,2026-06-30,5.252281,7.866104
3,Hyderabad,356,2025-02-19,2026-03-02,6.948157,9.486936
4,Bengaluru,466,2025-02-19,2026-06-30,7.599568,11.198896


In [43]:
mumbai_check = holdout_dfs['Mumbai'].copy()
mumbai_check['preds'] = np.expm1(trained_models['Mumbai'].predict(mumbai_check[feature_cols]))
mumbai_check['abs_error'] = (mumbai_check['pm25'] - mumbai_check['preds']).abs()

mumbai_check.sort_values('abs_error', ascending=False)[['date', 'pm25', 'preds', 'abs_error']].head(10)

,date,pm25,preds,abs_error
311,2026-03-10,493.0,83.130092,409.869908
248,2025-12-27,403.0,81.835903,321.164097
339,2026-04-10,336.0,57.253656,278.746344
64,2025-05-31,325.0,77.879201,247.120799
65,2025-06-01,325.0,84.118620,240.881380
63,2025-05-30,325.0,85.143149,239.856851
62,2025-05-29,266.0,35.186738,230.813262
411,2026-06-21,269.0,52.676448,216.323552
134,2025-08-09,235.0,23.940953,211.059047
342,2026-04-13,270.0,64.724903,205.275097


In [44]:
import joblib
import os

os.makedirs('/content/models', exist_ok=True)

for city, model in trained_models.items():
    joblib.dump(model, f'/content/models/{city}_lightgbm.joblib')

print(os.listdir('/content/models'))

['Mumbai_lightgbm.joblib', 'Bengaluru_lightgbm.joblib', 'Delhi_lightgbm.joblib', 'Hyderabad_lightgbm.joblib', 'Chennai_lightgbm.joblib']


In [45]:
!zip -r /content/models.zip /content/models
from google.colab import files
files.download('/content/models.zip')

  adding: content/models/ (stored 0%)
  adding: content/models/Mumbai_lightgbm.joblib (deflated 65%)
  adding: content/models/Bengaluru_lightgbm.joblib (deflated 65%)
  adding: content/models/Delhi_lightgbm.joblib (deflated 64%)
  adding: content/models/Hyderabad_lightgbm.joblib (deflated 65%)
  adding: content/models/Chennai_lightgbm.joblib (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
holdout_output_path = '/content/holdout_data'
os.makedirs(holdout_output_path, exist_ok=True)

for city, df in holdout_dfs.items():
    df.to_csv(f'{holdout_output_path}/{city}_holdout.csv', index=False)

print(os.listdir(holdout_output_path))

['Chennai_holdout.csv', 'Hyderabad_holdout.csv', 'Delhi_holdout.csv', 'Bengaluru_holdout.csv', 'Mumbai_holdout.csv']


In [47]:
!zip -r /content/holdout_data.zip /content/holdout_data
files.download('/content/holdout_data.zip')

  adding: content/holdout_data/ (stored 0%)
  adding: content/holdout_data/Chennai_holdout.csv (deflated 74%)
  adding: content/holdout_data/Hyderabad_holdout.csv (deflated 73%)
  adding: content/holdout_data/Delhi_holdout.csv (deflated 73%)
  adding: content/holdout_data/Bengaluru_holdout.csv (deflated 73%)
  adding: content/holdout_data/Mumbai_holdout.csv (deflated 73%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
predictions_output_path = '/content/predictions'
os.makedirs(predictions_output_path, exist_ok=True)

for city, df in holdout_dfs.items():
    preds_log = trained_models[city].predict(df[feature_cols])
    preds_actual = np.expm1(preds_log)

    out_df = pd.DataFrame({
        'date': df['date'],
        'actual': df['pm25'],
        'predicted': preds_actual
    })
    out_df.to_csv(f'{predictions_output_path}/{city}_predictions.csv', index=False)

print(os.listdir(predictions_output_path))

['Hyderabad_predictions.csv', 'Chennai_predictions.csv', 'Delhi_predictions.csv', 'Mumbai_predictions.csv', 'Bengaluru_predictions.csv']


In [49]:
!zip -r /content/predictions.zip /content/predictions
files.download('/content/predictions.zip')

  adding: content/predictions/ (stored 0%)
  adding: content/predictions/Hyderabad_predictions.csv (deflated 57%)
  adding: content/predictions/Chennai_predictions.csv (deflated 59%)
  adding: content/predictions/Delhi_predictions.csv (deflated 57%)
  adding: content/predictions/Mumbai_predictions.csv (deflated 57%)
  adding: content/predictions/Bengaluru_predictions.csv (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
importances = trained_models['Delhi'].booster_.feature_importance(importance_type='gain')
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
feat_imp.to_csv('/content/feature_importance.csv')
files.download('/content/feature_importance.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>